In [ ]:
import os
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# بارگذاری دیتابیس فصل قبل
embedding_function = OpenAIEmbeddings(model="text-embedding-3-small",base_url="https://api.tapsage.com/openai/v1",api_key="")
vector_db = Chroma(
    persist_directory="./chroma_db", 
    embedding_function=embedding_function,
    collection_name="my_course_collection"
)

# تبدیل به موتور جستجو (Retriever)
# k=3 یعنی همیشه ۳ تا سند مرتبط برگردان
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

In [ ]:
# الگوی پرامپت استاندارد RAG
template = """Answer the question based only on the following context:

{context}

Question: {question}

If you don't know the answer based on the context, simply say "I don't know". Do not make up an answer.
"""
prompt = ChatPromptTemplate.from_template(template)

# مدل زبانی
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0) # دمای صفر برای دقت بالا

In [ ]:
def format_docs(docs):
    """
    اسناد بازیابی شده را به یک رشته متنی طولانی تبدیل می‌کند.
    """
    return "\n\n".join([d.page_content for d in docs])

# تعریف زنجیره RAG
rag_chain = (
    # بخش اول: آماده‌سازی داده‌ها
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    |
    # بخش دوم: پرامپت
    prompt
    |
    # بخش سوم: مدل و پارسر
    llm
    | StrOutputParser()
)

In [ ]:
# تست سیستم
response = rag_chain.invoke("مریخ چه رنگی است؟") # سوالی که در دیتابیس فصل قبل داشتیم
print(response)

# تست عدم توهم
response_unknown = rag_chain.invoke("قیمت دلار امروز چقدر است؟")
print(response_unknown)